In [1]:
pip install wfdb torch transformers neurokit2 tslearn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.8/163.8 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 107.4 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 79.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 42.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 8.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 30.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 10.1 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 8.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 84.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━

In [2]:
import wfdb
import numpy as np
from sklearn.model_selection import train_test_split
import torch
import sys
import json
import matplotlib.pyplot as plt
from collections import defaultdict
sys.path.append('/kaggle/input/preprocessor')
sys.path.append('/kaggle/input/toksss')

from transformers import PreTrainedTokenizerFast, BigBirdForSequenceClassification, AutoModelForSequenceClassification
from torch.utils.data import Dataset, DataLoader
import ECGSignalPreprocessor  
import huggingface_compatible_tokenizer
import os

def load_ecg_signals(filepaths):
    all_data = []
    labels = []
    for path in filepaths:
        record = wfdb.rdrecord(path)
        labels.append(record.__dict__["comments"][2].replace("Diagnosis report:",""))
        signal = np.transpose(record.p_signal[:,0:12])
        x = np.linspace(0, 10, record.sig_len)
        
        processor = ECGSignalPreprocessor.ECGSignalProcessor(
            signal, 12, record.sig_len, record.sig_name, x)
        processor.detect_peaks_and_dips()
        processor.apply_bandpass_filter()
        processor.apply_notch_filter(processor.filtered_signal)
        processor.apply_savgol_filter(processor.filtered_signal)
        processor.apply_detrend(processor.smoothed)
        processor.trim_signal(processor.detrend)
        all_data.append(processor.trim_arr)
    return all_data, labels

# Padding
def pad_signal(signal, target_length=5000):
    signal = np.asarray(signal)
    return np.array([
        np.pad(lead, (0, max(0, target_length - len(lead))), mode='wrap')[:target_length]
        for lead in signal
    ])



#Filepaths
base_path = "/kaggle/input/ecg-data/wilson-central-terminal-ecg-database-1.0.1"
filepaths = []

for root, dirs, files in os.walk(base_path):
    for file in files:
        if file.endswith(".hea"):
            full_path = os.path.join(root, file)
            without_ext = full_path[:-4]  # remove ".hea"
            filepaths.append(without_ext)


all_data, labels = load_ecg_signals(filepaths)
padded_signals = [pad_signal(sig) for sig in all_data]
vocab_file={}
tok_model = huggingface_compatible_tokenizer.Word_Tokenizer(filepaths, vocab_file)
toks_sax, toks_sax_inv = tok_model.tokenize_sax(padded_signals) #Apply SAX
reshaped_toks = tok_model.reshape_tokens(toks_sax)  #Reshape the token arrays
toks2sentence = tok_model.token_to_string(reshaped_toks)  #Concatenate the leads and make sentenece

2025-07-30 11:48:21.736406: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1753876101.929969      36 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1753876101.988683      36 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
/usr/local/lib/python3.11/dist-packages/tslearn/utils/utils.py:108: UserWarning: 2-Dimensional data passed. Assuming these are 1 1-dimensional timeseries
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/tslearn/utils/utils.py:108: UserWarning: 2-Dimensional data passed. Assuming these are 1 1-dimensional timeseries
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/tslearn/utils/utils.py:108: UserWarning: 2-Dimensional 

In [3]:
#Label handeling
# Count labels
count_dict = defaultdict(int)
for label in labels:
    count_dict[label.strip()]+=1

print(count_dict)

#Remove labels occuring less than 15 times
valid_labels = {label for label, count in count_dict.items() if count >= 15}

filtered_samples = []
filtered_labels = []

for i in range(len(labels)):
    label = labels[i].strip()
    if label in valid_labels:
        filtered_samples.append(toks2sentence[i])
        filtered_labels.append(label)

#Label to id
unique_labels = sorted(set(filtered_labels))
label2id = {label: idx for idx, label in enumerate(unique_labels)}
label_id = [label2id[label.strip()] for label in filtered_labels]



num_labels = len(unique_labels)

count_dict_f = defaultdict(int)
for label in filtered_labels:
    count_dict_f[label]+=1

print(count_dict_f)

target_label = 'Non STsegmentelevation myocardial infarction (NSTEMI)'

final_samples = []
final_labels = []

for sample, label in zip(filtered_samples, filtered_labels):
    if label != target_label:
        final_samples.append(sample)
        final_labels.append(label)

final_label_ids = [label2id[label] for label in final_labels]
unique_labels_new = sorted(set(final_labels))

for label in sorted(unique_labels_new):
    print(f"Label: {label} | ID: {label2id[label]} | Frequency: {count_dict[label]}")
print(type(final_samples[0]))
print(f"Total dataset size after excluding NSTEMI: {len(final_samples)}")

defaultdict(<class 'int'>, {'STsegmentelevation myocardial infarction (STEMI)': 31, 'Ventricular tachycardia (VT)': 18, 'Non STsegmentelevation myocardial infarction (NSTEMI)': 100, 'Coronary artery disease': 23, 'Urosepsis': 4, 'Severe Mitral Stenosis': 17, 'Syncope unknown cause': 2, 'Atrial fibrillation': 13, 'Non STsegmentelevation myocardial infarction (NSTEMI)- rapid Atrial fibrillation': 1, 'Cardiomyopathy': 4, 'Stable angina underwent PCI (Percutaneous Coronary Intervention )': 8, 'not reported': 38, 'Rapid Atrial fibrillation - pericarditis': 1, 'Atypical chest pain': 31, 'Fall secondary to alcohol intoxication': 1, 'Stable angina': 29, 'Atrial flutter': 10, 'Gastritis (non cardiac chest pain)': 1, 'Supraventricular tachycardia (SVT)': 13, 'Congestive Cardic failure (CCF)': 3, 'Complete Heart block': 3, 'Inferior STEMI (STsegmentelevation myocardial infarction)': 3, 'Hypertrophic obstructive cardiomyopathy': 30, 'Pulmonary embolism.': 5, 'Congestive cardiac failure (CHF) exace

In [9]:
#Split train and eval
train_texts, val_texts, train_labels, val_labels = train_test_split(
    final_samples, final_label_ids, test_size=0.2, random_state=42)


train_dataset = tok_model.create_ecg_dataset(train_texts, train_labels)
val_dataset = tok_model.create_ecg_dataset(val_texts, val_labels)


vocab_size = tok_model.vocab_size()
print(vocab_size)

vocab = tok_model.get_vocab()
#print(vocab) --> 306

decoded_texts = []
inverse_signals = []
for i, text in enumerate(val_texts):

    sax_token = tok_model._tokenize(text)
    #print("SAXtoken:" + str(sax_token))

    token_to_id = [tok_model._convert_token_to_id(tok) for tok in sax_token]
    #print("token to id:" , token_to_id)

    id_to_token = [tok_model._convert_id_to_token(tok_id) for tok_id in token_to_id]
    #print("id to token:" , id_to_token)

    decode , inverse = tok_model._decode(token_to_id, skip_special_tokens = True)
    #print("decoded:" + decode)
    #print(len(inverse)) --> 5000
    decoded_texts.append(decode)
    inverse_signals.append(inverse)
print(type(inverse_signals[0][0])) #inverse_signals[0] -- np.array, inverse_signals[0][0] -- np.float64
#print(type(val_texts[0]))
train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=4)

for batch in train_loader:
    input_ids = batch['input_ids']
    attention_mask = batch['attention_mask']
    labels = batch['labels']
    print("Input IDs:", input_ids)
    print("Labels:", labels)
    break

#FIRST SPLIT TRAIN AND EVAL. THEN, USE SAX AND WORLD LEVEL TOKENIZER.
#FINALLY USE DATALOADER.
#SO THAT YOU CAN COMPARE THE DECODED ARRAY AND THE ORIGINAL ONE!!

306


/usr/local/lib/python3.11/dist-packages/tslearn/utils/utils.py:108: UserWarning: 2-Dimensional data passed. Assuming these are 1 1-dimensional timeseries
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/tslearn/utils/utils.py:108: UserWarning: 2-Dimensional data passed. Assuming these are 1 1-dimensional timeseries
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/tslearn/utils/utils.py:108: UserWarning: 2-Dimensional data passed. Assuming these are 1 1-dimensional timeseries
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/tslearn/utils/utils.py:108: UserWarning: 2-Dimensional data passed. Assuming these are 1 1-dimensional timeseries
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/tslearn/utils/utils.py:108: UserWarning: 2-Dimensional data passed. Assuming these are 1 1-dimensional timeseries
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/tslearn/utils/utils.py:108: UserWarning: 2-Dimensional data passed. Assuming these are 1 1-dimensional ti

<class 'numpy.float64'>
<class 'str'>
Input IDs: tensor([[137, 154, 163,  ..., 137, 141, 144],
        [237, 164, 122,  ..., 178, 150,  78],
        [168, 160, 153,  ..., 145, 124, 153],
        [ 50, 149, 205,  ..., 147, 172, 175]])
Labels: tensor([ 9, 10,  0,  6])


/usr/local/lib/python3.11/dist-packages/tslearn/utils/utils.py:108: UserWarning: 2-Dimensional data passed. Assuming these are 1 1-dimensional timeseries
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/tslearn/utils/utils.py:108: UserWarning: 2-Dimensional data passed. Assuming these are 1 1-dimensional timeseries
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/tslearn/utils/utils.py:108: UserWarning: 2-Dimensional data passed. Assuming these are 1 1-dimensional timeseries
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/tslearn/utils/utils.py:108: UserWarning: 2-Dimensional data passed. Assuming these are 1 1-dimensional timeseries
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/tslearn/utils/utils.py:108: UserWarning: 2-Dimensional data passed. Assuming these are 1 1-dimensional timeseries
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/tslearn/utils/utils.py:108: UserWarning: 2-Dimensional data passed. Assuming these are 1 1-dimensional ti

In [6]:
# Load BigBird model
model = BigBirdForSequenceClassification.from_pretrained("google/bigbird-roberta-base", num_labels=num_labels)
model.resize_token_embeddings(vocab_size)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

# Simple training loop
from torch.optim import AdamW

optimizer = AdamW(model.parameters(), lr=2e-5)
model.train()

for epoch in range(3):
    total_loss = 0
    for batch in train_loader:
        batch = {k: v.to(device) for k, v in batch.items()}
        outputs = model(**batch)
        loss = outputs.loss
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
        total_loss += loss.item()

    print(f"Epoch {epoch+1} - Loss: {total_loss:.4f}")

# Switch to evaluation mode
model.eval()
predictions = []
true_labels = []

with torch.no_grad():
    for batch in val_loader:
        inputs = {k: v.to(device) for k, v in batch.items() if k != "labels"}
        print("Batch label min/max:", batch["labels"].min(), batch["labels"].max())
        outputs = model(**inputs)
        preds = torch.argmax(outputs.logits, dim=1)
        predictions.extend(preds.cpu().tolist())
        true_labels.extend(batch["labels"].tolist())

print("Predictions:", predictions)
print("True labels:", true_labels)

from sklearn.metrics import classification_report

print(classification_report(true_labels, predictions))

config.json:   0%|          | 0.00/760 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/513M [00:00<?, ?B/s]

Some weights of BigBirdForSequenceClassification were not initialized from the model checkpoint at google/bigbird-roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


model.safetensors:   0%|          | 0.00/513M [00:00<?, ?B/s]

Attention type 'block_sparse' is not possible if sequence_length: 512 <= num global tokens: 2 * config.block_size + min. num sliding tokens: 3 * config.block_size + config.num_random_blocks * config.block_size + additional buffer: config.num_random_blocks * config.block_size = 704 with config.block_size = 64, config.num_random_blocks = 3. Changing attention type to 'original_full'...


Epoch 1 - Loss: 146.9505
Epoch 2 - Loss: 132.1624
Epoch 3 - Loss: 115.9525
Batch label min/max: tensor(0) tensor(11)
Batch label min/max: tensor(3) tensor(11)
Batch label min/max: tensor(4) tensor(10)
Batch label min/max: tensor(2) tensor(9)
Batch label min/max: tensor(4) tensor(11)
Batch label min/max: tensor(0) tensor(9)
Batch label min/max: tensor(0) tensor(11)
Batch label min/max: tensor(2) tensor(11)
Batch label min/max: tensor(1) tensor(8)
Batch label min/max: tensor(2) tensor(11)
Batch label min/max: tensor(0) tensor(11)
Batch label min/max: tensor(2) tensor(11)
Batch label min/max: tensor(1) tensor(4)
Batch label min/max: tensor(1) tensor(10)
Batch label min/max: tensor(0) tensor(3)
Batch label min/max: tensor(9) tensor(9)
Predictions: [9, 2, 2, 2, 2, 2, 6, 2, 2, 2, 11, 1, 1, 4, 2, 4, 2, 11, 2, 6, 4, 2, 2, 2, 2, 2, 11, 2, 4, 1, 2, 2, 2, 2, 2, 2, 2, 2, 2, 6, 2, 2, 2, 2, 2, 2, 2, 2, 11, 2, 2, 1, 2, 2, 2, 1, 2, 2, 2, 2, 4]
True labels: [9, 0, 2, 11, 10, 3, 4, 11, 8, 7, 10, 4, 6, 9

/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
